# Module 2: Ensemble Methods — Hands-On Exercise
## Machine learning for energy applications — Module 2, Exercise 1

In this exercise, we will explore **ensemble methods** — techniques that combine multiple models
to produce stronger predictions than any single model alone. We will work with the classic Iris
dataset and build progressively more powerful classifiers.

---
## Overview

**Topics covered:**

1. Data preparation (Iris dataset, label mapping, stratified train/test split)
2. Single Decision Tree baseline
3. Random Forest classifier
4. Gradient Boosting classifier
5. Model comparison with cross-validation on the training rows, then one final test evaluation
6. Feature importance: impurity (MDI) versus permutation importance
7. Building an sklearn Pipeline (StandardScaler + model), saving and reloading it
8. Hyperparameter tuning with GridSearchCV
9. (Optional) XGBoost, or scikit-learn's HistGradientBoostingClassifier

**Key idea:** A single decision tree is prone to overfitting, but by aggregating many trees —
either through *bagging* (Random Forest) or *boosting* (Gradient Boosting) — we can often
improve generalization. Whether they do on a particular dataset is an empirical question,
answered with validation evidence and the spread across folds.

**Evaluation rules in this notebook:**
- Models are fitted, compared and tuned on the 120 **training** rows only, with 5-fold
  cross-validation (the same folds for every model).
- The 30 **test** rows are opened only in clearly labelled cells, after the models are fixed:
  the final evaluation of Lab 1 (end of Part 5), the permutation importance of the already
  evaluated forest (Part 6) and the final evaluation of Lab 2 (end of Part 8). No model,
  setting or feature is chosen from them.

In [ ]:
import tempfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, classification_report

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['figure.dpi'] = 100

---
## Part 1: Data Preparation

Load the Iris dataset from `sklearn.datasets`, inspect its structure, print the label mapping,
and split into training (80%) and test (20%) sets.

The split is stratified so that each class keeps its share in both parts. From here on the
**test rows are sealed**: we do not score them until the final evaluation at the end of Part 5.

In [ ]:
# Load the Iris dataset using load_iris()
# YOUR CODE HERE
# Hint: iris = load_iris()
#        X = iris.data, y = iris.target
#        feature_names = iris.feature_names, target_names = iris.target_names


# Print the dataset shape, number of features, feature names,
# and the label mapping (integer code -> class name)
# YOUR CODE HERE
# Hint: {i: str(name) for i, name in enumerate(target_names)}


# Print the class distribution (how many samples per class)
# YOUR CODE HERE


# Create a train/test split (80/20) with stratify=y and random_state=42
# The test rows stay sealed until the final evaluation at the end of Part 5
# YOUR CODE HERE
# Hint: X_train, X_test, y_train, y_test = train_test_split(...)


# Print the number of training and test samples
# YOUR CODE HERE

---
## Part 2: Single Decision Tree

Train a single `DecisionTreeClassifier` on the training rows as a baseline. This will help us
see how much improvement the ensemble methods provide.

We do not score the test rows here. Training accuracy only shows how well a model fits the rows
it has seen; the models are compared with cross-validation on the training rows in Part 5.

In [ ]:
# Train a DecisionTreeClassifier with random_state=42 on the training rows
# YOUR CODE HERE
# Hint: dt_clf = DecisionTreeClassifier(random_state=42)
#        dt_clf.fit(X_train, y_train)


# Calculate and print the TRAINING accuracy (predict on X_train) using accuracy_score
# Why is this number optimistic? (The test rows stay sealed until the end of Part 5.)
# YOUR CODE HERE


# Print the depth and the number of leaves of the tree
# Hint: dt_clf.get_depth(), dt_clf.get_n_leaves()
# YOUR CODE HERE

---
## Part 3: Random Forest

Train a `RandomForestClassifier` with 100 estimators on the training rows. Random Forest is a
*bagging* method: it trains many decision trees on bootstrap samples of the rows, with a random
subset of features considered at each split, then averages the trees' class probabilities (a soft vote).

In [ ]:
# Train a RandomForestClassifier with n_estimators=100 and random_state=42 on the training rows
# YOUR CODE HERE
# Hint: rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)


# Calculate and print the training accuracy
# YOUR CODE HERE

---
## Part 4: Gradient Boosting

Train a `GradientBoostingClassifier` with 100 estimators on the training rows. Gradient Boosting
is a *boosting* method: it trains trees sequentially, where each new tree focuses on correcting
the errors made by the previous ensemble.

In [ ]:
# Train a GradientBoostingClassifier with n_estimators=100 and random_state=42 on the training rows
# YOUR CODE HERE
# Hint: gb_clf = GradientBoostingClassifier(n_estimators=100, random_state=42)


# Calculate and print the training accuracy
# YOUR CODE HERE

---
## Part 5: Model Comparison

Compare all three models using **5-fold cross-validation on the training rows only**
(`X_train`, `y_train`). The test rows must stay out of the folds: a score computed with the
test rows inside the folds cannot also be reported as independent test evidence.

With a classifier, `cv=5` means `StratifiedKFold(n_splits=5)` without shuffling, so every model
is scored on the same five folds and we can compare them fold by fold. We will visualize the
results with a bar chart.

In [ ]:
# Create a dictionary of models: 'Decision Tree', 'Random Forest', 'Gradient Boosting'
# YOUR CODE HERE
# Hint: models = {
#     'Decision Tree': DecisionTreeClassifier(random_state=42),
#     'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
#     'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
# }


# Loop over models and compute 5-fold cross-validation scores on the TRAINING rows only
# (the test rows must stay out of the folds)
# Store results in a dictionary: cv_results[name] = scores
# Print the mean and std for each model
# YOUR CODE HERE
# Hint: scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')


# Print the accuracy of every model in every fold (e.g. pd.DataFrame(cv_results))
# Are the differences between models larger than the differences between folds?
# YOUR CODE HERE


# Create a bar chart comparing mean CV accuracies with error bars (std)
# YOUR CODE HERE
# Hint: use plt.bar() with yerr for error bars
#        use capsize=5 for the error bar caps
#        set plt.ylim(0.85, 1.02) for a good view

### 5.1 Extension (optional): out-of-bag accuracy and a validation curve

For fast groups; both checks use the training rows only.
- **Out-of-bag (OOB) accuracy** scores each training row with the trees whose bootstrap sample
  left it out. Like ordinary cross-validation, it assumes the rows are unrelated.
- **Validation curve for boosting:** split the training rows into training and validation rows,
  then plot validation accuracy against the number of boosting stages with `staged_predict`.

In [ ]:
# Extension (optional)
# 1. Fit RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42) on the
#    training rows and compare its oob_score_ with the forest's CV mean from Part 5
# YOUR CODE HERE


# 2. Split the TRAINING rows into training and validation rows, for example
#    X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.25,
#                                                  random_state=42, stratify=y_train)
#    Fit GradientBoostingClassifier(n_estimators=100, random_state=42) on X_fit, y_fit and plot
#    validation accuracy against the number of stages
# YOUR CODE HERE
# Hint: val_acc = [accuracy_score(y_val, y_hat) for y_hat in model.staged_predict(X_val)]

### 5.2 Final evaluation of Lab 1: open the test rows once

First write down your conclusion from the cross-validation (which model, or "no clear
difference"). Then score the three models from Parts 2–4, each fitted on all 120 training
rows, on the 30 sealed test rows — once. One test row is worth 1/30 = 3.3 percentage points,
so compare any difference with the fold spread above. Nothing is re-tuned or re-chosen after
this cell.

In [ ]:
# FINAL EVALUATION (Lab 1): run this once, after writing down your CV-based conclusion.
# For each of dt_clf, rf_clf and gb_clf (fitted in Parts 2-4 on the training rows):
#   - predict on X_test; compute the test accuracy and the number of correct test rows
#   - print the classification report (use target_names=target_names)
# Collect CV mean, CV std, test accuracy and correct rows in one table (a pandas DataFrame)
# YOUR CODE HERE

---
## Part 6: Feature Importance

Tree-based ensembles provide **impurity importance** (mean decrease in impurity, MDI) in
`feature_importances_`. It is computed from the training splits and rewards any feature the
trees happened to split on. **Permutation importance** instead measures how much a held-out
score drops when one feature is shuffled: the reliance of a fitted model on that feature.

The forest from Part 3 is already fixed and was evaluated at the end of Part 5, so here we may
use the test rows to *interpret* it. With 30 rows the values are noisy, and no feature may be
dropped or selected on the basis of this table. Neither measure is a causal effect.

In [ ]:
# Extract feature importances from the trained Random Forest (rf_clf)
# YOUR CODE HERE
# Hint: importances = rf_clf.feature_importances_
#        indices = np.argsort(importances)


# Create a horizontal bar chart of feature importances
# YOUR CODE HERE
# Hint: use plt.barh() with the sorted importances
#        use plt.yticks() with the corresponding feature names
#        set xlabel, title, and call plt.tight_layout()


# Print the feature importance values in descending order
# YOUR CODE HERE

In [ ]:
# Compute permutation importance of rf_clf on the test rows (interpretation of the fixed model)
# and put it next to the MDI values in one table (a pandas DataFrame indexed by feature name)
# YOUR CODE HERE
# Hint: perm = permutation_importance(rf_clf, X_test, y_test, scoring='accuracy',
#                                     n_repeats=20, random_state=0)
#        use perm.importances_mean and perm.importances_std

---
## Part 7: sklearn Pipeline

An sklearn `Pipeline` chains preprocessing and modelling steps into a single object.
This ensures that all transformations (e.g. scaling) are applied consistently during
both training and prediction, and prevents data leakage during cross-validation: the scaler
is refitted inside every training fold.

We will build a pipeline that:
1. Scales the features with `StandardScaler`
2. Classifies with `RandomForestClassifier`

For trees, scaling changes nothing (splits depend only on the order of the values); we keep the
scaler to practise packaging preprocessing with the model. We cross-validate the pipeline on the
training rows, then save it with `joblib`, reload it, and check that the reloaded pipeline
reproduces the predictions on five reference rows.

In [ ]:
# Create a Pipeline with two steps:
#   1. ('scaler', StandardScaler())
#   2. ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
# YOUR CODE HERE
# Hint: pipe = Pipeline([('scaler', StandardScaler()), ('clf', RandomForestClassifier(...))])


# Print the pipeline steps
# YOUR CODE HERE


In [ ]:
# Fit the pipeline on the training rows
# YOUR CODE HERE
# Hint: pipe.fit(X_train, y_train)


# Run 5-fold cross-validation of the pipeline on the TRAINING rows and print the results
# Compare with the Random Forest CV score from Part 5: did the scaler change anything?
# YOUR CODE HERE
# Hint: cv_pipe = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')

In [ ]:
# Save the fitted pipeline with joblib.dump to a file (for example in tempfile.mkdtemp())
# Keep a reference batch of five rows and the pipeline's predictions and probabilities on it
# YOUR CODE HERE
# Hint: model_path = Path(tempfile.mkdtemp()) / 'iris_rf_pipeline.joblib'
#        joblib.dump(pipe, model_path)
#        reference_rows = X_test[:5]
#        reference_pred = pipe.predict(reference_rows)
#        reference_proba = pipe.predict_proba(reference_rows)

In [ ]:
# Reload the pipeline with joblib.load (do not refit it) and assert that it reproduces
# the reference predictions and probabilities exactly
# YOUR CODE HERE
# Hint: reloaded = joblib.load(model_path)
#        assert np.array_equal(reloaded.predict(reference_rows), reference_pred)

---
## Part 8: Hyperparameter Tuning with GridSearchCV

Use `GridSearchCV` to search over a grid of hyperparameters for the Random Forest
inside the pipeline. When referencing pipeline parameters, use the format
`stepname__parametername` (double underscore).

We will tune:
- `clf__n_estimators`: \[50, 100, 200\]
- `clf__max_depth`: \[None, 5, 10\]
- `clf__min_samples_split`: \[2, 5\]

The search selects on 5-fold cross-validation of the **training rows**. In `cv_results_`,
"test" (as in `mean_test_score`) means the held-out fold of each split, i.e. validation data,
not our sealed test rows. Inspect every row: how many settings share the best score?

In [ ]:
# Define the parameter grid for the pipeline
# YOUR CODE HERE
# Hint: param_grid = {
#     'clf__n_estimators': [50, 100, 200],
#     'clf__max_depth': [None, 5, 10],
#     'clf__min_samples_split': [2, 5]
# }


In [ ]:
# Create and run GridSearchCV with the pipeline, param_grid, cv=5, scoring='accuracy'
# on the TRAINING rows
# YOUR CODE HERE
# Hint: grid_search = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy',
#                                   verbose=1, n_jobs=-1)
#        grid_search.fit(X_train, y_train)


# Print the best parameters and best cross-validation score
# YOUR CODE HERE
# Hint: grid_search.best_params_, grid_search.best_score_

In [ ]:
# Put grid_search.cv_results_ in a DataFrame and show the three parameters with
# mean_test_score, std_test_score and rank_test_score, sorted by rank_test_score
# ("test" here means the held-out CV fold, not our sealed test rows)
# How many settings share the best score?
# YOUR CODE HERE
# Hint: results = pd.DataFrame(grid_search.cv_results_)

### 8.1 Final evaluation of Lab 2: the locked choice on the test rows

`GridSearchCV` has refitted the selected setting on all training rows (`best_estimator_`).
That choice is now locked, and it is scored once on the sealed test rows. Nothing is tuned after
this cell; if you wanted to try another grid, the test score would no longer be an independent
estimate.

In [ ]:
# FINAL EVALUATION (Lab 2): score the locked choice once on the test rows
# YOUR CODE HERE
# Hint: best_model = grid_search.best_estimator_
#        y_pred_best = best_model.predict(X_test)


# Print the test accuracy and classification report for the best model
# YOUR CODE HERE

---
## Part 9: Advanced — XGBoost (Optional)

XGBoost is a popular, high-performance gradient boosting library. It is an optional package
(listed in `requirements-labs-optional.txt` at the course root). If it is installed, we compare
it with the scikit-learn models; otherwise the cell uses scikit-learn's built-in
`HistGradientBoostingClassifier`.

XGBoost 2 and later ignore `use_label_encoder` (with a warning), so the cell leaves it out:
the labels must already be integers 0, 1, 2 (as here), and `eval_metric` is set in the constructor.
The comparison uses the same 5 folds of the training rows as Part 5; the sealed test rows are
not reopened for this extra model.

In [ ]:
# Optional: XGBoost if installed, otherwise scikit-learn's HistGradientBoostingClassifier.
# Compare it with the Part 5 models using 5-fold CV on the TRAINING rows (test rows stay sealed).
# YOUR CODE HERE
# Hint:
# try:
#     from xgboost import XGBClassifier
#     booster = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3,
#                             eval_metric='mlogloss', random_state=42)
# except ImportError:
#     from sklearn.ensemble import HistGradientBoostingClassifier
#     print("xgboost is not installed; using HistGradientBoostingClassifier.")
#     booster = HistGradientBoostingClassifier(learning_rate=0.1, max_depth=3, random_state=42)
# scores = cross_val_score(booster, X_train, y_train, cv=5, scoring='accuracy')

---
## Summary

In this exercise we:

1. **Loaded and explored** the Iris dataset (150 samples, 4 features, 3 classes), printed the
   label mapping, and made a stratified 80/20 split with the test rows sealed.
2. Trained a **single Decision Tree** as a baseline classifier.
3. Trained a **Random Forest** (bagging) and **Gradient Boosting** (boosting) classifier.
4. **Compared** all three models with 5-fold cross-validation on the training rows (same folds),
   then opened the test rows once for the final evaluation of Lab 1.
5. Compared **impurity (MDI) and permutation importance** for the fixed Random Forest.
6. Built an sklearn **Pipeline** combining `StandardScaler` and `RandomForestClassifier`, saved it,
   reloaded it, and checked that it reproduces its predictions.
7. Used **GridSearchCV** on the training rows, inspected every row of `cv_results_`, and scored
   the locked choice once on the test rows.
8. (Optional) Compared **XGBoost** (or `HistGradientBoostingClassifier`) on the same folds.

**Key takeaways:**
- Bagging averages away much of the variance of deep trees, and boosting builds a flexible
  model from many small corrections; but on a small, nearly separable dataset such as Iris the differences between models are within the fold-to-fold
  noise. Check the spread before ranking models.
- Choose models and settings with cross-validation on the training rows (or a validation
  split); open the test rows once, for the locked choice.
- MDI is a quick training-data diagnostic; permutation importance on held-out rows measures how
  much a fitted model relies on a feature. Neither is a causal effect.
- sklearn Pipelines keep preprocessing inside every cross-validation fold and travel with the
  model when it is saved; check that a reloaded model reproduces its predictions.
- GridSearchCV selects on cross-validation scores; look at `cv_results_` for ties before trusting
  `best_params_`.